In [1]:
!pip install bert-score

     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     ---------------------------------------- 57.7/57.7 kB 3.0 MB/s eta 0:00:00
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 41.5/41.5 kB ? eta 0:00:00
   ---------------------------------------- 0.0/61.1 kB ? eta -:--:--
   ---------------------------------------- 61.1/61.1 kB 3.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.5/114.6 MB 9.4 MB/s eta 0:00:13
   ---------------------------------------- 1.0/114.6 MB 10.3 MB/s eta 0:00:11
    --------------------------------------- 1.6/114.6 MB 11.5 MB/s eta 0:00:10
    --------------------------------------- 2.3/114.6 MB 12.2 MB/s eta 0:00:10
    --------------------------------------- 2.8/114.6 MB 12.1 MB/s eta 0:00:10
   - -------------------------------------- 3.5/114.6 MB 12.3 MB/s eta 0:00:10
   - ------


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
results_df_local=pd.read_csv('/content/drive/MyDrive/FINETUNING/generated_summaries_local.csv')

ModuleNotFoundError: No module named 'google'

In [3]:
y_pred_local=results_df_local['generated_summary'].tolist()
y_true_local=results_df_local['true_summary'].tolist()

In [4]:
from bert_score import score

print("Calculating BertScore...")
# Calculate BertScore
# You can specify a different model type if needed, e.g., model_type="bert-base-uncased"
# Using lang='en' will automatically select an appropriate model for English.
P, R, F1 = score(y_pred_local, y_true_local, lang="en", verbose=True)

print(f"\nSystem level BertScore:")
print(f"Precision: {P.mean().item():.4f}")
print(f"Recall: {R.mean().item():.4f}")
print(f"F1 Score: {F1.mean().item():.4f}")

# Add BertScore results to the DataFrame
results_df_local['bert_score_precision'] = P.tolist()
results_df_local['bert_score_recall'] = R.tolist()
results_df_local['bert_score_f1'] = F1.tolist()

print("\nFirst 5 rows of the DataFrame with BertScores:")
display(results_df_local.head())

Calculating BertScore...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 2.70 seconds, 41.78 sentences/sec

System level BertScore:
Precision: 0.8449
Recall: 0.8583
F1 Score: 0.8514

First 5 rows of the DataFrame with BertScores:


,document_id,original_document,true_summary,generated_summary,bert_score_precision,bert_score_recall,bert_score_f1
0,38295789,The ex-Reading defender denied fraudulent trad...,Former Premier League footballer Sam Sodje has...,The prosecution alleges that the alleged tradi...,0.855569,0.857665,0.856616
1,40202028,Voges was forced to retire hurt on 86 after su...,Middlesex batsman Adam Voges will be out until...,The county team has a strong batting lineup wi...,0.850086,0.833974,0.841953
2,36177725,Seven photographs taken in the Norfolk country...,The Duchess of Cambridge will feature on the c...,The shoot was a significant investment for the...,0.857812,0.889227,0.873237
3,35751255,"Chris Poole - known as ""moot"" online - created...",Google has hired the creator of one of the web...,.\n```,0.766476,0.816054,0.790488
4,35275743,Four police officers were injured in the incid...,Two teenagers have been charged in connection ...,The boy's parents have reported the incident t...,0.868817,0.891135,0.879835


In [3]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00


In [5]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d746b7f970dfdccfb5c1e15f7144d8e29fdc1a35eab63c954f45f91bc4770e55
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [10]:
import evaluate

print("Calculating ROUGE scores...")

# Load the ROUGE metric
rouge = evaluate.load("rouge")

# Compute ROUGE scores
# Using the same y_pred_local and y_true_local from the previous cell
results_rouge = rouge.compute(predictions=y_pred_local, references=y_true_local)

print("\nSystem level ROUGE scores:")
print(f"ROUGE-1: {results_rouge['rouge1']:.4f}")
print(f"ROUGE-2: {results_rouge['rouge2']:.4f}")
print(f"ROUGE-L: {results_rouge['rougeL']:.4f}")
print(f"ROUGE-Lsum: {results_rouge['rougeLsum']:.4f}")

Calculating ROUGE scores...

System level ROUGE scores:
ROUGE-1: 0.1461
ROUGE-2: 0.0218
ROUGE-L: 0.1109
ROUGE-Lsum: 0.1237


In [13]:
!pip install summac

In [17]:
!pip install summac --upgrade

In [4]:
!pip install deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.6/864.6 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.3/228.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.6 MB/s eta 0:00:00


In [37]:
!pip install --upgrade huggingface_hub


In [5]:
from huggingface_hub import login

# You will be prompted to enter your Hugging Face token
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [15]:
!pip install transformers accelerate bitsandbytes


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import pandas as pd
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
local_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    quantization_config=quantization_config,
    device_map="auto"
)

local_tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [ ]:
!pip install deepeval
import os
import torch
import gc # Garbage collector

# Ensure there are no other models loaded, if possible
torch.cuda.empty_cache()
gc.collect()

# This helps PyTorch manage fragmented memory efficiently
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import pandas as pd
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.metrics import ContextualRelevancyMetric, FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval import evaluate

# 1. Define the Custom DeepEval Model to wrap our already loaded gemma-3-1b-it model
class CustomGemmaJudge(DeepEvalBaseLLM):
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def load_model(self):
        # The model is already loaded, so we just return it
        return self.model

    def generate(self, prompt: str) -> str:
        # Ensure garbage collection before generation
        torch.cuda.empty_cache()
        gc.collect()

        # Print the device to confirm GPU usage
        print(f"CustomGemmaJudge using device: {self.model.device}")

        # Truncate prompt if too long to prevent memory issues during inference
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4000).to(self.model.device)

        # Generate response with controlled parameters for judge output
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=256,  # Judge output needs to be concise
            num_beams=1,         # Keep greedy decoding for deterministic judge output
            do_sample=False,     # No sampling for judge
            temperature=0.0,     # Low temperature for deterministic output
            top_p=1.0
        )

        input_length = inputs.input_ids.shape[1]
        response = self.tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        # Clean up immediately after generating
        del inputs, outputs
        torch.cuda.empty_cache()

        return response

    async def a_generate(self, prompt: str) -> str:
        # DeepEval requires an async wrapper
        return self.generate(prompt)

    def get_model_name(self):
        return "gemma-3-1b-it-judge"

# 2. Initialize your new Judge with the already loaded local_model and local_tokenizer
# Ensure local_model and local_tokenizer are available from previous cells (e.g., ae96f025)
print("Initializing local Gemma model as DeepEval Judge...")
custom_gemma_judge = CustomGemmaJudge(model=local_model, tokenizer=local_tokenizer)

# 3. Setup the Summarization Metrics using your custom local judge
faithfulness_metric = FaithfulnessMetric(threshold=0.7, model=custom_gemma_judge)
contextual_relevancy_metric = ContextualRelevancyMetric(threshold=0.7, model=custom_gemma_judge)
answer_relevancy_metric = AnswerRelevancyMetric(threshold=0.7, model=custom_gemma_judge)

# 4. Load your CSV data (assuming results_df_local is the correct DataFrame)
print("Loading dataset...")
df = pd.read_csv('/content/drive/MyDrive/FINETUNING/generated_summaries_local.csv')

# Prepare test cases for deepeval
test_cases = []
for index, row in df.iterrows():
    test_case = LLMTestCase(
        input=row['original_document'],  # The document is the input to the summarizer
        actual_output=row['generated_summary'], # The generated summary is the actual output
        expected_output=row['true_summary'], # The true summary is what we expect
        retrieval_context=[row['original_document']] # Context for metrics like contextual relevancy
    )
    test_cases.append(test_case)

print(f"Prepared {len(test_cases)} test cases for Deepeval evaluation.")

# 5. Execute Evaluation
print("Starting Deepeval evaluation with local Gemma model (this will take a while)...")
evaluate(test_cases, [faithfulness_metric, contextual_relevancy_metric, answer_relevancy_metric])
print("Deepeval evaluation with local Gemma model complete!")

# 6. Safely Extract and Print Results
print(f"\nAverage Faithfulness: {faithfulness_metric.overall_score:.4f}")
print(f"Average Contextual Relevancy: {contextual_relevancy_metric.overall_score:.4f}")
print(f"Average Answer Relevancy: {{answer_relevancy_metric.overall_score:.4f}}")

# Example of printing individual results (optional)
for i, test_case in enumerate(test_cases):
    print(f"\n--- Test Case {i+1} ---")
    print(f"Document: {test_case.input[:100]}...")
    print(f"Generated Summary: {test_case.actual_output[:100]}...")
    print(f"True Summary: {test_case.expected_output[:100]}...")
    print(f"Faithfulness Score: {faithfulness_metric.measure(test_case).score:.4f}")
    print(f"Contextual Relevancy Score: {contextual_relevancy_metric.measure(test_case).score:.4f}")
    print(f"Answer Relevancy Score: {answer_relevancy_metric.measure(test_case).score:.4f}")


Initializing local Gemma model as DeepEval Judge...
Loading dataset...
Prepared 113 test cases for Deepeval evaluation.
Starting Deepeval evaluation with local Gemma model (this will take a while)...


✨ You're running DeepEval's latest Faithfulness Metric! (using gemma-3-1b-it-judge, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gemma-3-1b-it-judge, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gemma-3-1b-it-judge, strict=False, 
async_mode=True)...

Output()

...

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


CustomGemmaJudge using device: cuda:0

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


CustomGemmaJudge using device: cuda:0

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
